# Preparación de datos

**Conjunto de datos:** Dataset 2 - Lugares de emisiones

**Nombre de archivo:** emission_permits_anom_2.json

## 0. Inicialización

Instalar ydata-profiling

In [34]:
!pip install ydata-profiling

Instalar geopy

In [35]:
!pip install geopy

Instalar folium

In [36]:
!pip install folium

Importaciones

In [37]:
import pandas as pd
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport
import seaborn as sns

import folium
import json
from tabulate import tabulate

Visualización de tablas y gráficas

In [38]:
sns.set_style("darkgrid")

def print_table(df):
    print(tabulate(df, headers='keys', tablefmt='simple_outline'))

Lectura y muestra del archivo

In [39]:
# 1. Cargar el JSON
with open('../data/original/emission_permits_anom_2.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 2. Extraer features
features = data['features']

# 3. Construir DataFrame con properties y coordenadas
df = pd.DataFrame([
    {
        **feature['properties'],
        'Latitud': feature['geometry']['coordinates'][1],
        'Longitud': feature['geometry']['coordinates'][0]
    }
    for feature in features
])

df.head()

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Cuenca,Latitud,Longitud
0,73640.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,CENTRO,None,Otros,Horno,Río Bogotá,4.703418,-74.226561
1,73788.0,Seguimiento y Control,Ubate,Cundinamarca,LENGUAZAQUE,Resguardo,None,Carbón,Caldera Horno,Río Suárez,5.318407,-73.704281
2,74314.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,MADRID,LA PUNTA,None,ACPM,Caldera Horno,Río Bogotá,4.800462,-74.210355
3,75972.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,Balsillas,None,Fuel Oil No.8,Planta de Asfalto,Río Bogotá,4.678797,-74.284112
4,78824.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,FUNZA,El Hato,None,Carbón,Caldera Horno,Río Bogotá,4.699590,-74.193752


**1.** Transformar IDExpediente a integer

In [40]:
df['IDExpediente'] = df['IDExpediente'].astype("Int64")

**2.** Eliminar columnas innecesarias

In [41]:
df = df.drop(columns=['Cuenca'])

**3.** Eliminar duplicados

In [42]:
df = df.drop_duplicates()

**4.** Manejar la capitalización en las variables categóricas

In [43]:
df["Vereda"] = df["Vereda"].str.strip().str.upper()
df["TipoFuenteEmision"] = df["TipoFuenteEmision"].str.strip().str.capitalize()
df.head()

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,Longitud
0,73640,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,CENTRO,None,Otros,Horno,4.703418,-74.226561
1,73788,Seguimiento y Control,Ubate,Cundinamarca,LENGUAZAQUE,RESGUARDO,None,Carbón,Caldera horno,5.318407,-73.704281
2,74314,Seguimiento y Control,Sabana Occidente,Cundinamarca,MADRID,LA PUNTA,None,ACPM,Caldera horno,4.800462,-74.210355
3,75972,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,BALSILLAS,None,Fuel Oil No.8,Planta de asfalto,4.678797,-74.284112
4,78824,Seguimiento y Control,Sabana Occidente,Cundinamarca,FUNZA,EL HATO,None,Carbón,Caldera horno,4.699590,-74.193752


**5.** Mejorar la presentación de los (sin definir)

In [44]:
df["TipoCombustible"] = df["TipoCombustible"].replace("(sin definir)", "Sin definir")
df["TipoFuenteEmision"] = df["TipoFuenteEmision"].replace("(sin definir)", "Sin definir")
df[(df["TipoCombustible"] == "Sin definir") | (df["TipoFuenteEmision"] == "Sin definir")]

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,Longitud
42,138622,Seguimiento y Control,Sabana Centro,Cundinamarca,NEMOCON,PATIO BONITO,None,Sin definir,Sin definir,5.118908,-73.895435
43,139320,Seguimiento y Control,Ubate,Cundinamarca,TAUSA,RASGATÁ,None,Sin definir,Horno,5.190613,-73.879585
61,150306,Seguimiento y Control,Sabana Centro,Cundinamarca,NEMOCON,PATIO BONITO,None,Sin definir,Sin definir,5.125421,-73.904398
100,171196,Seguimiento y Control,Bogotá y Municipio de la Calera,Distrito Capital,LOCALIDAD DE CIUDAD BOLIVAR,MOCHUELO BAJO,None,Sin definir,Sin definir,4.506914,-74.150135
102,171204,Seguimiento y Control,Bogotá y Municipio de la Calera,Distrito Capital,LOCALIDAD DE CIUDAD BOLIVAR,MOCHUELO BAJO,None,Sin definir,Sin definir,4.516030,-74.149127
...,...,...,...,...,...,...,...,...,...,...,...
539,291446,Sancionatorio,Chiquinquira,Boyacá,RAQUIRA,CASCO URBANO,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,12.629099,-39.893628
540,291712,Sancionatorio,Sumapaz,Cundinamarca,ARBELAEZ,SAN ROQUE,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.274013,-74.446186
541,292030,Sancionatorio,Ubate,Cundinamarca,CUCUNUBA,PUEBLO VIEJO,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,5.228254,-73.817856
542,292322,Sancionatorio,Alto Magdalena,Cundinamarca,GIRARDOT,URBANO,Emitir por encima de los parámetros establecid...,Sin definir,Sin definir,11.736503,-41.043145


**6.** Estandarizar la vereda

In [45]:
df["Vereda"] = df["Vereda"].replace({
    "CASCO URBANO": "AREA URBANA",
    "URBANO": "AREA URBANA",
    "CENTRO URBANO": "AREA URBANA",
})

**7.** Estandarizar los tipos de fuente

In [46]:
df["TipoFuenteEmision"].value_counts()

TipoFuenteEmision
Sin definir                            326
Horno                                  131
Caldera                                 24
Caldera horno                           14
Secadores                                7
Planta de asfalto                        6
Molino                                   3
Chimenea 1                               3
Trituradora                              2
Noaplica (área de operación)             2
Reactor                                  1
Planta de asfalto adm                    1
Barrilado de grafito                     1
Filtro molino pendular                   1
Aspiración molino danioni i              1
Triturador de escombros                  1
Campana de extracción  plomo 1           1
Horno de secado                          1
Horno arcillas de soacha tipo túnel      1
Triturador de material                   1
Horno túnel 1 soacha 2                   1
Chimenea triunfo central                 1
700 bhp vr2                         

In [47]:
df["TipoFuenteEmision"] = df["TipoFuenteEmision"].replace({
    "Chimenea 1": "Chimenea",
    "Noaplica (área de operación)": "No aplica (área de operación)",
    "Planta de asfalto adm": "Planta de asfalto",
    "Barrilado de grafito": "Horno",
    "Filtro molino pendular": "Molino",
    "Aspiración molino danioni i": "Molino",
    "Triturador de escombros": "Trituradora",
    "Campana de extracción  plomo 1": "Horno",
    "Horno de secado": "Horno",
    "Horno arcillas de soacha tipo túnel": "Horno",
    "Triturador de material": "Trituradora",
    "Horno túnel 1 soacha 2": "Horno",
    "Chimenea triunfo central": "Chimenea",
    "700 bhp vr2": "Caldera",
    "Planta de mezcla asfáltica": "Planta de asfalto",
    "Batería de coquización a": "Batería de coquización",
    "Molino buhler": "Molino",
    "Planta trituradora": "Trituradora",
    "Batería de producción de coque": "Batería de coquización",
})

In [48]:
df["TipoFuenteEmision"].value_counts()

TipoFuenteEmision
Sin definir                      326
Horno                            136
Caldera                           25
Caldera horno                     14
Planta de asfalto                  8
Secadores                          7
Molino                             6
Trituradora                        5
Chimenea                           4
No aplica (área de operación)      2
Batería de coquización             2
Reactor                            1
Name: count, dtype: int64

**8.** Estandarizar los tipos de combustible

In [49]:
df["TipoCombustible"].value_counts()

TipoCombustible
Sin definir      327
Carbón           128
Otros             26
Gas               20
ACPM              13
NoAplica           7
Fuel Oil No.8      5
Coque              5
Mezcla             2
Madera             1
Leña               1
Hulla              1
Name: count, dtype: int64

In [50]:
df["TipoCombustible"] = df["TipoCombustible"].replace("NoAplica", "No aplica")

In [51]:
df["TipoCombustible"].value_counts()

TipoCombustible
Sin definir      327
Carbón           128
Otros             26
Gas               20
ACPM              13
No aplica          7
Fuel Oil No.8      5
Coque              5
Mezcla             2
Madera             1
Leña               1
Hulla              1
Name: count, dtype: int64

**9.** Estandarizar los municipios

In [52]:
df["Municipio"] = df["Municipio"].replace("LOC.USAQUEN CERROS ORIENTALES", "LOCALIDAD DE USAQUEN")

In [53]:
df["Municipio"] = df["Municipio"].str.replace("LOCALIDAD DE ", "")

In [54]:
print_table(pd.DataFrame(df["Municipio"].value_counts()))

┌─────────────────────┬─────────┐
│ Municipio           │   count │
├─────────────────────┼─────────┤
│ SOACHA              │      56 │
│ CIUDAD BOLIVAR      │      52 │
│ NEMOCON             │      39 │
│ COGUA               │      33 │
│ GIRARDOT            │      25 │
│ CUCUNUBA            │      23 │
│ TAUSA               │      18 │
│ GUACHETA            │      18 │
│ SUTATAUSA           │      15 │
│ RAQUIRA             │      15 │
│ MOSQUERA            │      14 │
│ SIBATE              │      14 │
│ LENGUAZAQUE         │      14 │
│ CAJICA              │      13 │
│ TOCANCIPA           │      12 │
│ FUSAGASUGA          │      12 │
│ VILLAPINZON         │      11 │
│ ZIPAQUIRA           │      10 │
│ COTA                │      10 │
│ FACATATIVA          │       9 │
│ SIMIJACA            │       9 │
│ VILLETA             │       8 │
│ UBATE               │       7 │
│ MADRID              │       7 │
│ RICAURTE            │       6 │
│ CHIQUINQUIRA        │       6 │
│ FUNZA       

**10.** Llenar nulos en class con 'Sin sanción'

In [55]:
df["Class"] = df["Class"].fillna("Sin sanción")

**11.** Corregir empresas con localización inválida, usando la información geográfica dada

In [56]:
# from geopy.geocoders import Nominatim
# from geopy.extra.rate_limiter import RateLimiter

# # --- 1. Crear geolocalizador ---
# geolocator = Nominatim(user_agent="geo_colombia")
# geocode = RateLimiter(geolocator.geocode, min_delay_seconds=2, max_retries=2, error_wait_seconds=2.0)  # para no exceder límites

# # --- 2. Filtrar las filas mal ubicadas ---
# df_mal_ubicacion = df[(df['Latitud'] >= 12.27) | (df['Longitud'] >= -66.50)].copy()

# print(f"Fuentes con mala ubicación: {len(df_mal_ubicacion)}")

# # --- 3. Función para obtener coordenadas ---
# def obtener_coordenadas_fila(row):
#     # Construimos una descripción geográfica más completa
#     lugar_vereda = f"{row['Vereda']}, {row['Municipio']}, {row['Departamento']}, Colombia"
#     lugar_municipio = f"{row['Municipio']}, {row['Departamento']}, Colombia"
    
#     try:
#         location = geocode(lugar_vereda)
#         if location:
#             return pd.Series([location.latitude, location.longitude, "Media"])
#         else:
#             # Si no encuentra la vereda, probar con municipio
#             location = geocode(lugar_municipio)
#             if location:
#                 if row['Vereda'] == "AREA URBANA":
#                     return pd.Series([location.latitude, location.longitude, "Media"])
#                 else:
#                     return pd.Series([location.latitude, location.longitude, "Baja"])

#     except Exception as e:
#         print(f"Error en {lugar_vereda}: {e}")
    
#     return pd.Series([None, None, None])

# # --- 4. Aplicar función ---
# df_mal_ubicacion[['Latitud', 'Longitud', 'PrecisionUbicacion']] = (
#     df_mal_ubicacion.apply(obtener_coordenadas_fila, axis=1)
# )

# df_mal_ubicacion

In [57]:
# df_mal_ubicacion.to_csv("archivos_generados/df_mal_ubicacion_corregido.csv", index=False)

In [58]:
df_mal_ubicacion = pd.read_csv("archivos_generados/df_mal_ubicacion_corregido.csv", index_col="Unnamed: 0")

In [59]:
import numpy as np
from geopy.distance import distance
from geopy import Point
import pandas as pd

def mover_puntos_geodesico_en_subdf(subdf, desplazamiento_metros=100):
    """
    Para un subdataframe de un Municipio-Vereda, desplaza
    aleatoriamente los puntos que comparten la misma Latitud/Longitud.
    """
    subdf = subdf.copy()

    # Cambiar esto:
    duplicados_mask = subdf.duplicated(subset=["Latitud", "Longitud"], keep=False)

    # si no hay duplicados, devolvemos el subdf sin cambios
    if not duplicados_mask.any():
        return subdf

    # iterar por cada conjunto de coordenadas duplicadas
    for (lat0, lon0), grp in subdf[duplicados_mask].groupby(["Latitud", "Longitud"]):
        if grp.shape[0] <= 1:
            continue

        n = grp.shape[0]
        angles = np.random.uniform(0, 360, n)
        radios = np.random.uniform(desplazamiento_metros * 0.3, desplazamiento_metros, n)

        nuevas = []
        for r, a in zip(radios, angles):
            destino = distance(meters=r).destination(Point(lat0, lon0), a)
            nuevas.append((destino.latitude, destino.longitude))

        # asignar nuevas coordenadas al índice correcto
        latitudes = [t[0] for t in nuevas]
        longitudes = [t[1] for t in nuevas]
        subdf.loc[grp.index, "Latitud"] = latitudes
        subdf.loc[grp.index, "Longitud"] = longitudes

    return subdf

# --- Procesar por grupos de Municipio y Vereda sin perder columnas ---
resultado_parts = []
for (mun, ver), sub in df_mal_ubicacion.groupby(["Municipio", "Vereda"]):
    # opcional: asegurar que las columnas existan en el subdf
    sub = sub.copy()
    sub["Municipio"] = mun
    sub["Vereda"] = ver

    processed = mover_puntos_geodesico_en_subdf(sub)
    resultado_parts.append(processed)

df_mal_ubicacion = pd.concat(resultado_parts, ignore_index=False)


In [60]:
df_mal_ubicacion

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,Longitud,PrecisionUbicacion
476,261630,Sancionatorio,Chiquinquira,Boyacá,CHIQUINQUIRA,AREA URBANA,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,5.618273,-73.816748,Media
523,282348,Sancionatorio,Bogotá y Municipio de la Calera,Distrito Capital,CIUDAD BOLIVAR,MOCHUELO ALTO,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.488357,-74.148341,Media
221,281510,En Trámite,Bogotá y Municipio de la Calera,Distrito Capital,CIUDAD BOLIVAR,MOCHUELO BAJO,NaN,Sin definir,Sin definir,4.507824,-74.148079,Media
524,282350,Sancionatorio,Bogotá y Municipio de la Calera,Distrito Capital,CIUDAD BOLIVAR,MOCHUELO BAJO,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.508538,-74.148380,Media
525,282352,Sancionatorio,Bogotá y Municipio de la Calera,Distrito Capital,CIUDAD BOLIVAR,MOCHUELO BAJO,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.508888,-74.148648,Media
...,...,...,...,...,...,...,...,...,...,...,...,...
481,265736,Sancionatorio,Ubate,Cundinamarca,TAUSA,RASGATA ALTO,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,5.194927,-73.956308,Baja
532,283576,Sancionatorio,Alto Magdalena,Cundinamarca,TOCAIMA,MORRO AZUL,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,4.457927,-74.634223,Baja
96,170762,Seguimiento y Control,Sabana Centro,Cundinamarca,TOCANCIPA,VERGANZO,NaN,Carbón,Caldera,4.970039,-73.974057,Media
498,272438,Sancionatorio,Bogotá y Municipio de la Calera,Distrito Capital,USAQUEN,EL PÁRAMO,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.695219,-74.030932,Baja


**Nota:** Si presenta el error `Make this Notebook Trusted to load map: File -> Trust Notebook` y usa Anaconda, use la Anaconda Prompt para dirigirse a la carpeta `preparacion` y escriba ```jupyter trust "Dataset 2 - Fuentes de emisiones.ipynb"```

In [61]:
min_lat = df_mal_ubicacion['Latitud'].min()
max_lat = df_mal_ubicacion['Latitud'].max()
min_lon = df_mal_ubicacion['Longitud'].min()
max_lon = df_mal_ubicacion['Longitud'].max()

bounds = [
    [min_lat, min_lon],  # esquina inferior izquierda x
    [min_lat, max_lon],  # inferior derecha
    [max_lat, max_lon],  # superior derecha
    [max_lat, min_lon],  # superior izquierda
]

m = folium.Map(location=[4.71, -74.07], zoom_start=5)

# Dibujar el rectángulo
folium.Polygon(
    locations=bounds,
    color='blue',
    weight=2,
    fill=True,
    fill_opacity=0.1
).add_to(m)

# Agregar marcadores de cada ubicación del dataset
for _, row in df_mal_ubicacion.iterrows():
    color = 'red' if row['PrecisionUbicacion'] == 'Baja' else 'blue'
    folium.Marker(
        location=[row['Latitud'], row['Longitud']],
        tooltip=f"{row['IDExpediente']} - {row['Municipio']}, {row['Vereda']}, Precisión: {row['PrecisionUbicacion']}",
        icon=folium.Icon(color=color)
    ).add_to(m)

# Límites geográficos reales de Colombia
real_max_lat = 12.27
real_max_lon = -66.50

# Marcas adicionales para los límites geográficos de Colombia
folium.PolyLine(
    locations=[[real_max_lat, min_lon], [real_max_lat, real_max_lon]],
    color='green',
    weight=2,
    tooltip=f"Máxima latitud de Colombia terrestre: {real_max_lat}"
).add_to(m)

folium.PolyLine(
    locations=[[min_lat, real_max_lon], [real_max_lat, real_max_lon]],
    color='green',
    weight=2,
    tooltip=f"Mínima longitud de Colombia: {real_max_lon}"
).add_to(m)

m

In [62]:
# --- 5. Crear la columna con valor constante en el df original ---
df['PrecisionUbicacion'] = 'Alta'

# --- 6. Actualizar df original con los valores corregidos ---
cols_to_update = ['Latitud', 'Longitud', 'PrecisionUbicacion']

df.update(df_mal_ubicacion[cols_to_update])
df

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,Longitud,PrecisionUbicacion
0,73640,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,CENTRO,Sin sanción,Otros,Horno,4.703418,-74.226561,Alta
1,73788,Seguimiento y Control,Ubate,Cundinamarca,LENGUAZAQUE,RESGUARDO,Sin sanción,Carbón,Caldera horno,5.318407,-73.704281,Alta
2,74314,Seguimiento y Control,Sabana Occidente,Cundinamarca,MADRID,LA PUNTA,Sin sanción,ACPM,Caldera horno,4.800462,-74.210355,Alta
3,75972,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,BALSILLAS,Sin sanción,Fuel Oil No.8,Planta de asfalto,4.678797,-74.284112,Alta
4,78824,Seguimiento y Control,Sabana Occidente,Cundinamarca,FUNZA,EL HATO,Sin sanción,Carbón,Caldera horno,4.699590,-74.193752,Alta
...,...,...,...,...,...,...,...,...,...,...,...,...
539,291446,Sancionatorio,Chiquinquira,Boyacá,RAQUIRA,AREA URBANA,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,5.496933,-73.625257,Media
540,291712,Sancionatorio,Sumapaz,Cundinamarca,ARBELAEZ,SAN ROQUE,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.274013,-74.446186,Alta
541,292030,Sancionatorio,Ubate,Cundinamarca,CUCUNUBA,PUEBLO VIEJO,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,5.228254,-73.817856,Alta
542,292322,Sancionatorio,Alto Magdalena,Cundinamarca,GIRARDOT,AREA URBANA,Emitir por encima de los parámetros establecid...,Sin definir,Sin definir,4.303889,-74.803207,Media


**12.** Obtener probabilidades de incidencia de las empresas

In [63]:
fuel_matrix = pd.read_csv('../data/enriquecida/air_fuel_matrix.csv')
source_matrix = pd.read_csv('../data/enriquecida/air_source_matrix.csv')

In [64]:
# Nos aseguramos de que los nombres de columnas coincidan
fuel_matrix = fuel_matrix.rename(columns={'Tipo de combustible': 'TipoCombustible'})
source_matrix = source_matrix.rename(columns={'Tipo de fuente': 'TipoFuenteEmision'})

for var in fuel_matrix['Variable'].unique():
    # Subconjuntos para esta variable específica
    fuel_sub = (
        fuel_matrix[fuel_matrix['Variable'] == var]
        .rename(columns={
            'Probabilidad': f'Probabilidad_Fuel_{var}',
            'Ponderación': f'Ponderacion_Fuel_{var}'
        })[['TipoCombustible', f'Probabilidad_Fuel_{var}', f'Ponderacion_Fuel_{var}']]
    )
    
    src_sub = (
        source_matrix[source_matrix['Variable'] == var]
        .rename(columns={
            'Probabilidad': f'Probabilidad_Source_{var}',
            'Ponderación': f'Ponderacion_Source_{var}'
        })[['TipoFuenteEmision', f'Probabilidad_Source_{var}', f'Ponderacion_Source_{var}']]
    )
    
    # Merges sobre df
    df = df.merge(fuel_sub, on='TipoCombustible', how='left')
    df = df.merge(src_sub, on='TipoFuenteEmision', how='left')
    
    # Calcular probabilidad
    df[f'ProbabilidadIncidencia_{var}'] = (
        (df[f'Probabilidad_Fuel_{var}'] * df[f'Ponderacion_Fuel_{var}'] +
         df[f'Probabilidad_Source_{var}'] * df[f'Ponderacion_Source_{var}']) / 9
    )
    
    # Eliminar las columnas intermedias
    df = df.drop(columns=[
        f'Probabilidad_Fuel_{var}', 
        f'Ponderacion_Fuel_{var}',
        f'Probabilidad_Source_{var}', 
        f'Ponderacion_Source_{var}'
    ])

Resultado final

In [65]:
df

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,Longitud,PrecisionUbicacion,ProbabilidadIncidencia_SO2,ProbabilidadIncidencia_PM2.5,ProbabilidadIncidencia_NO2,ProbabilidadIncidencia_CO,ProbabilidadIncidencia_NO,ProbabilidadIncidencia_PM10,ProbabilidadIncidencia_NOX
0,73640,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,CENTRO,Sin sanción,Otros,Horno,4.703418,-74.226561,Alta,0.511111,0.800000,0.711111,0.577778,0.711111,0.800000,0.755556
1,73788,Seguimiento y Control,Ubate,Cundinamarca,LENGUAZAQUE,RESGUARDO,Sin sanción,Carbón,Caldera horno,5.318407,-73.704281,Alta,0.755556,0.800000,0.688889,0.644444,0.688889,0.800000,0.733333
2,74314,Seguimiento y Control,Sabana Occidente,Cundinamarca,MADRID,LA PUNTA,Sin sanción,ACPM,Caldera horno,4.800462,-74.210355,Alta,0.955556,0.800000,0.888889,0.644444,0.888889,0.800000,0.933333
3,75972,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,BALSILLAS,Sin sanción,Fuel Oil No.8,Planta de asfalto,4.678797,-74.284112,Alta,0.866667,0.777778,0.777778,0.600000,0.777778,0.844444,0.822222
4,78824,Seguimiento y Control,Sabana Occidente,Cundinamarca,FUNZA,EL HATO,Sin sanción,Carbón,Caldera horno,4.699590,-74.193752,Alta,0.755556,0.800000,0.688889,0.644444,0.688889,0.800000,0.733333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,291446,Sancionatorio,Chiquinquira,Boyacá,RAQUIRA,AREA URBANA,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,5.496933,-73.625257,Media,0.311111,0.355556,0.266667,0.266667,0.266667,0.422222,0.311111
532,291712,Sancionatorio,Sumapaz,Cundinamarca,ARBELAEZ,SAN ROQUE,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.274013,-74.446186,Alta,0.311111,0.355556,0.266667,0.266667,0.266667,0.422222,0.311111
533,292030,Sancionatorio,Ubate,Cundinamarca,CUCUNUBA,PUEBLO VIEJO,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,5.228254,-73.817856,Alta,0.311111,0.355556,0.266667,0.266667,0.266667,0.422222,0.311111
534,292322,Sancionatorio,Alto Magdalena,Cundinamarca,GIRARDOT,AREA URBANA,Emitir por encima de los parámetros establecid...,Sin definir,Sin definir,4.303889,-74.803207,Media,0.311111,0.355556,0.266667,0.266667,0.266667,0.422222,0.311111


In [66]:
reporte = ProfileReport(df)
reporte.to_file("archivos_generados/Reporte perfilamiento - Dataset 2 Final.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 19/19 [00:00<00:00, 113.36it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Exportar a CSV

In [67]:
df.to_csv('../data/preparada/emission_permits.csv', index=False)